# Librairies 📚

In [1]:
# =========================
# Standard library
# =========================
import os
import sys
import time
import math
import re
import importlib

# =========================
# Data / scientific stack
# =========================
import numpy as np
import pandas as pd

# =========================
# Visualisation
# =========================
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

# =========================
# Stats
# =========================
import scipy.stats as stats
from scipy.stats import kruskal
import pingouin as pg
from statsmodels.stats.outliers_influence import variance_inflation_factor

from dython.nominal import (
    associations,
    correlation_ratio,  # eta²
    cramers_v,          # V de Cramér
    theils_u            # Alternative asymétrique
)

# =========================
# Machine Learning
# =========================
import lightgbm as lgb

from xgboost import XGBRegressor

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error,
    r2_score,
)
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    LabelEncoder,
    MultiLabelBinarizer,
    OneHotEncoder,
    RobustScaler,
    StandardScaler,
)
from sklearn.svm import SVR

# =========================
# Local imports (src/)
# =========================
sys.path.append(os.path.abspath(".."))  # si notebook dans /notebooks

import src.outliers_function as of
import src.association_report_function as ar
import src.analyse_bivariee as ab
import src.modeles_comparator as mdl
import src.tunning_parameter as tp
import src.outliers_treatment as ot

importlib.reload(of)

# =========================
# Pandas display options
# =========================
pd.set_option("display.max_columns", None)
pd.options.display.float_format = "{:,.2f}".format

# Import de la table 🪑

In [2]:
df_backup = pd.read_csv("/home/maxime/projects/building-energy-Seattle/data/processed/data_to_use.csv")
df_backup.drop("Unnamed: 0", axis=1, inplace=True)
df_backup.head(2)

,OSEBuildingID,BuildingType,PrimaryPropertyType,PropertyName,Address,ZipCode,TaxParcelIdentificationNumber,CouncilDistrictCode,Neighborhood,Latitude,Longitude,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFATotal,ListOfAllPropertyUseTypes,LargestPropertyUseType,ENERGYSTARScore,SiteEnergyUse(kBtu),SteamUse(kBtu),Electricity(kBtu),NaturalGas(kBtu),TotalGHGEmissions,nb_certifications,nb_property_uses,is_mixed_use,building_age,buildings_gfa_ratio,parking_gfa_ratio,largest_use_ratio,gfa_per_building,gfa_per_floor,age_bin,log_gfa,has_gas,has_electricity,has_steam,gas_prop,elec_prop,steam_prop
0,1,NonResidential,Hotel,Mayflower park hotel,405 Olive way,"98,101.00",0659000030,7,DOWNTOWN,47.61,-122.34,1927,1.00,12,88434,Hotel,Hotel,60.00,"7,226,362.50","2,003,882.00","3,946,027.00","1,276,453.00",249.98,0,1,0,89,1.00,0.00,1.00,"88,434.00","7,369.50",80+,11.39,1,1,1,0.18,0.55,0.28
1,2,NonResidential,Hotel,Paramount Hotel,724 Pine street,"98,101.00",0659000220,7,DOWNTOWN,47.61,-122.33,1996,1.00,11,103566,"Hotel, Parking, Restaurant",Hotel,61.00,"8,387,933.00",0.00,"3,242,851.00","5,145,082.00",295.86,0,3,1,20,0.85,0.15,0.81,"103,566.00","9,415.09",0-20,11.55,1,1,0,0.61,0.39,0.00


In [3]:
df = df_backup.copy()
df["CouncilDistrictCode"] = df["CouncilDistrictCode"].astype("category")
df["ZipCode"] = df["ZipCode"].astype("category")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1529 entries, 0 to 1528
Data columns (total 40 columns):
 #   Column                         Non-Null Count  Dtype   
---  ------                         --------------  -----   
 0   OSEBuildingID                  1529 non-null   int64   
 1   BuildingType                   1529 non-null   str     
 2   PrimaryPropertyType            1529 non-null   str     
 3   PropertyName                   1529 non-null   str     
 4   Address                        1529 non-null   str     
 5   ZipCode                        1516 non-null   category
 6   TaxParcelIdentificationNumber  1529 non-null   str     
 7   CouncilDistrictCode            1529 non-null   category
 8   Neighborhood                   1529 non-null   str     
 9   Latitude                       1529 non-null   float64 
 10  Longitude                      1529 non-null   float64 
 11  YearBuilt                      1529 non-null   int64   
 12  NumberofBuildings              1529 non-null 

# Etape 1 - Preprocessing

## Traitement des outliers

## Séparation des colonnes catégoriques et nuémriques

In [4]:
df.select_dtypes(include="number").columns

Index(['OSEBuildingID', 'Latitude', 'Longitude', 'YearBuilt',
       'NumberofBuildings', 'NumberofFloors', 'PropertyGFATotal',
       'ENERGYSTARScore', 'SiteEnergyUse(kBtu)', 'SteamUse(kBtu)',
       'Electricity(kBtu)', 'NaturalGas(kBtu)', 'TotalGHGEmissions',
       'nb_certifications', 'nb_property_uses', 'is_mixed_use', 'building_age',
       'buildings_gfa_ratio', 'parking_gfa_ratio', 'largest_use_ratio',
       'gfa_per_building', 'gfa_per_floor', 'log_gfa', 'has_gas',
       'has_electricity', 'has_steam', 'gas_prop', 'elec_prop', 'steam_prop'],
      dtype='str')

In [5]:
# J'enlève la certification pour le moment
numeric_features = [
            'Latitude',
            'Longitude',

            'YearBuilt',

            'NumberofBuildings',
            'NumberofFloors',

            'PropertyGFATotal', # Je le garde et j'utiliserai le log_gfa ensuite

            'ENERGYSTARScore',

            'nb_certifications',
            'nb_property_uses',

            "building_age",

            'buildings_gfa_ratio',
            'parking_gfa_ratio',

            'gas_prop',
            'elec_prop',
            'steam_prop'
]

In [15]:
categorical_features = [
    'BuildingType',
    'PrimaryPropertyType',
    'ZipCode',
    'CouncilDistrictCode',
    'LargestPropertyUseType',
    'ListOfAllPropertyUseTypes'
]

## Séparation en test train split

In [16]:
col_sel = numeric_features + categorical_features
X = df[col_sel]
y_energy = df["SiteEnergyUse(kBtu)"] # df["log_SiteEnergyUse"]

y_ghg = df["TotalGHGEmissions"] # df["log_TotalGHG"]

In [17]:
X = X.copy()
X.columns = X.columns.astype(str)
numeric_features = [c for c in map(str, numeric_features) if c in X.columns]
categorical_features = [c for c in map (str, categorical_features) if c in X.columns]
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 1529 entries, 0 to 1528
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   Latitude                   1529 non-null   float64 
 1   Longitude                  1529 non-null   float64 
 2   YearBuilt                  1529 non-null   int64   
 3   NumberofBuildings          1529 non-null   float64 
 4   NumberofFloors             1529 non-null   int64   
 5   PropertyGFATotal           1529 non-null   int64   
 6   ENERGYSTARScore            981 non-null    float64 
 7   nb_certifications          1529 non-null   int64   
 8   nb_property_uses           1529 non-null   int64   
 9   building_age               1529 non-null   int64   
 10  buildings_gfa_ratio        1529 non-null   float64 
 11  parking_gfa_ratio          1529 non-null   float64 
 12  gas_prop                   1529 non-null   float64 
 13  elec_prop                  1529 non-null   f

In [18]:
X_train, X_test, y_train_energy, y_test_energy = train_test_split(
    X, y_energy, test_size=0.2, random_state=42
)

_, _, y_train_ghg, y_test_ghg = train_test_split(
    X, y_ghg, test_size=0.2, random_state=42
)

## Preprocessing

In [19]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")), # Pas de valeurs manquantes dans le modèle (je le garde pour template)
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")), # Pas de valeurs manquantes dans le modèle (je le garde pour template)
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
], remainder="drop")

# Dummy Model

## Energy model

In [20]:
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test  = X_test.replace([np.inf, -np.inf], np.nan)

In [22]:
from sklearn.linear_model import LinearRegression

energy_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", DummyRegressor())
])

energy_model.fit(X_train, y_train_energy)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers c

## GHG model – Régression linéaire

In [23]:
ghg_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LinearRegression())
])

ghg_model.fit(X_train, y_train_ghg)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers c

## Evaluation de Energy model

In [28]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_log_error

y_pred_energy = energy_model.predict(X_test)

print("Energy R2:", r2_score(y_test_energy, y_pred_energy))
print("Energy MAE:", mean_absolute_error(y_test_energy, y_pred_energy))
print("Energy RMSLE:", mean_squared_log_error(y_test_energy, y_pred_energy))

Energy R2: -0.002923849788805555
Energy MAE: 7221487.005418407
Energy RMSLE: 2.5046946825125738


## Evaluation de GHG model

In [29]:
y_pred_ghg = ghg_model.predict(X_test)

print("GHG R2:", r2_score(y_test_ghg, y_pred_ghg))
print("GHG MAE:", mean_absolute_error(y_test_ghg, y_pred_ghg))
print("Energy RMSLE:", mean_squared_log_error(y_test_ghg, y_pred_ghg))

GHG R2: 0.01851322506710229
GHG MAE: 137.74537344271516


ValueError: Mean Squared Logarithmic Error cannot be used when targets contain values less than or equal to -1.

In [ ]:
y_pred_ghg_real = np.exp(y_pred_ghg)
y_test_ghg_real = np.exp(y_test_ghg)

print("GHG R2 (real):", r2_score(y_test_ghg_real, y_pred_ghg_real))
print("GHG MAE (real):", mean_absolute_error(y_test_ghg_real, y_pred_ghg_real))

GHG R2 (real): -0.6259840232127338
GHG MAE (real): 101.47163504039067


In [ ]:
rmsle = np.sqrt(mean_squared_log_error(y_test_ghg_real, y_pred_ghg_real))
print("Energy RMSLE:", rmsle)

Energy RMSLE: 0.9269829110616719


Le modèle GHG explique correctement la structure générale des données en log-space (r2: 0.720).

Cependant, en échelle réelle, la performance chute fortement (R² = 0,10),

La variabilité est plus importante que pour l’énergie.

Cela suggère que les émissions de GES sont plus instables et probablement influencées par des variables non observées (mix énergétique réel, comportement des occupants, performance des équipements, etc.).

## Conclusion

Les modèles linéaires constituent une baseline solide :

Ils confirment que les caractéristiques structurelles des bâtiments expliquent une part importante de la performance énergétique.

Ils mettent également en évidence une forte hétérogénéité entre bâtiments.

Les erreurs restent importantes à l’échelle individuelle.

Ces résultats justifient le recours à des modèles non linéaires (Gradient Boosting, Random Forest, Dummy Mean...) afin d’améliorer la précision.

# Etape 5 - Comparaison des modèles

## Cross Validation sur les différents modèles

In [697]:
seed = 42
models_with_grids = [
    ("dummy_mean",
     DummyRegressor(strategy="mean"),
     {}),

    ("linear_regression",
     LinearRegression(),
     {}),

    ("elastic_net",
     ElasticNet(random_state=seed),
     {"model__alpha": [0.01, 0.1, 1.0],
      "model__l1_ratio": [0.2, 0.5, 0.8]}),

    ("svr",
     SVR(),
     {"model__C": [1, 10],
      "model__epsilon": [0.1, 0.2],
      "model__kernel": ["rbf"]}),

    ("random_forest",
     RandomForestRegressor(random_state=seed, n_jobs=-1),
     {"model__n_estimators": [200, 500],
      "model__max_depth": [None, 10, 20],
      "model__min_samples_split": [2, 5]}),

    ("lgbm",
     lgb.LGBMRegressor(random_state=seed),
     {"model__n_estimators": [200, 500],
      "model__learning_rate": [0.05, 0.1],
      "model__num_leaves": [31, 63]}),

    # ✅ XGBoost en régression (pas classifier)
    ("xgboost",
     XGBRegressor(
         random_state=seed,
         objective="reg:squarederror",
         n_jobs=-1
     ),
     {"model__n_estimators": [300, 600],
      "model__max_depth": [3, 6],
      "model__learning_rate": [0.05, 0.1]})
]

In [ ]:
df_compare = tp.tune_and_compare_models(
    models_with_grids=models_with_grids,
    preprocessor=preprocessor,
    X_train=X_train, y_train=y_train_energy,
    X_test=X_test, y_test=y_test_energy,
    cv=5,
    seed=42
)

df_compare

Durée d'exécution de tune_model: 1.272s
Durée d'exécution de tune_model: 1.112s


/home/maxime/miniconda3/envs/ds/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.976e+16, tolerance: 1.860e+13
  model = cd_fast.enet_coordinate_descent(
/home/maxime/miniconda3/envs/ds/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.874e+16, tolerance: 1.965e+13
  model = cd_fast.enet_coordinate_descent(
/home/maxime/miniconda3/envs/ds/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider incre

Durée d'exécution de tune_model: 1.486s
Durée d'exécution de tune_model: 0.663s
Durée d'exécution de tune_model: 13.974s
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.867871 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 979
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.815812 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 982
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.855691 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 982
[LightGBM] [Info] Number of data points in the train set: 978, number of used features: 53
[LightGBM] 

/home/maxime/miniconda3/envs/ds/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/maxime/miniconda3/envs/ds/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.116346 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 983
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of data points in the train set: 978, number of used features: 54
[LightGBM] [Info] Start training from score 7294355.579475
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/home/maxime/miniconda3/envs/ds/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.941222 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 980
[LightGBM] [Info] Number of data points in the train set: 978, number of used features: 53
[LightGBM] [Info] Start training from score 7249128.861580
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

/home/maxime/miniconda3/envs/ds/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/maxime/miniconda3/envs/ds/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.054832 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 982
[LightGBM] [Info] Number of data points in the train set: 979, number of used features: 53
[LightGBM] [Info] Start training from score 7597762.105856
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

## Comparaison des modèles

In [ ]:
baseline_energy = mdl.compare_resultat_pipeline(X_train, y_train_energy,preprocessor=preprocessor)
baseline_energy


In [ ]:
baseline_ghg = mdl.compare_resultat_pipeline(X_train, y_train_ghg,preprocessor=preprocessor)
baseline_ghg

### Optimisation et interprétation du modèle

A réaliser :
* Reprennez le meilleur algorithme que vous avez sécurisé via l'étape précédente, et réalisez une GridSearch de petite taille sur au moins 3 hyperparamètres.
* Si le meilleur modèle fait partie de la famille des modèles à arbres (RandomForest, GradientBoosting) alors utilisez la fonctionnalité feature importance pour identifier les features les plus impactantes sur la performance du modèle. Sinon, utilisez la méthode Permutation Importance de sklearn.

In [ ]:
# CODE OPTIMISATION ET INTERPRETATION DU MODELE